# ARC-v0.28 — NQ-GTE H=50 A100-Batched Long-Horizon Agentic-IR Bridge Audit

**Purpose.** Stress-test whether the frozen representation-versus-search-effort approximation-feedback ordering survives under substantially longer, recursive retrieval-feedback trajectories, while using an A100-friendly batched FAISS execution path.

This is a controlled bridge toward current **Agentic IR / Agentic RAG** settings, where retrieval is repeatedly interleaved with an evolving state. It is **not** a full LLM-agent experiment: there is no planner, adaptive tool-selection policy, generation module, memory manager, or learned stopping policy.

The v0.27 NQ-GTE severity match is reused unchanged:

- dataset: BEIR Natural Questions
- encoder: `thenlper/gte-small`
- representation contrast: IVF-PQ32 → IVF-SQ8 at `nprobe=64`
- search-effort contrast: IVF-SQ8 `nprobe=2 → 64`
- v0.27 FIT relative one-shot nDCG@10 gap mismatch: **2.3946%**
- validation membership: the same 1,726 v0.27 validation queries

## Frozen two-tier design

### Tier A — full-policy comparability
All 44 policies are evaluated under anchored and recursive feedback through **H=8**.

### Tier B — long-horizon Agentic-IR bridge
A structurally selected, outcome-independent 8-policy subset is evaluated through:

\[
T \in \{4,8,12,16,24,32,40,50\}.
\]

For every `alpha ∈ {0.1,0.3,0.5,0.7}`, the long-horizon subset contains:

- `mean-k20`
- `softmax-k20-t0.1`

The subset is fixed by policy structure before any new validation trajectory is generated.

## Primary new estimand

> **recursive operator, H=50, long-horizon subset, nDCG@10 H3abs: representation − severity-matched search effort**

Inference uses a 10,000-replicate paired query bootstrap.

Primary support criterion: the 95% CI is strictly above zero.

## Secondary pre-specified analyses

- recursive H=12/16/24/32/40;
- anchored H=12/16/24/32/40/50;
- full-policy anchored/recursive H=4/H=8;
- mean-only / softmax-only / equal-family estimates;
- MRR@10 and Recall@10 H3abs;
- H8→H50 persistence / saturation / reversal;
- raw round-8→round-50 absolute-gap growth;
- peak-gap round distribution.

## Execution backend

The notebook prefers **CUDA FAISS on a single A100** and batches all queries in a checkpoint through the same policy/operator/round. It automatically falls back to **batched CPU FAISS** if the GPU backend cannot faithfully restore both IVF-PQ and IVF-SQ8 indexes.

A FIT-only CPU-vs-GPU semantic audit is performed before validation trajectories. No validation setting is retuned.

Negative, null, saturation, late reversal, or over-extension results are retained unchanged.

In [ ]:
# Cell 1 — Robust A100 / FAISS setup
import os, sys, subprocess

def run(cmd):
    return subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

gpu_present = run(["bash", "-lc", "nvidia-smi -L"]).returncode == 0

run([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "faiss-cpu", "faiss-gpu", "faiss-gpu-cu12",
])

common = [
    "sentence-transformers",
    "pyarrow",
    "scipy",
    "tqdm",
    "psutil",
    "requests",
]

if gpu_present:
    print("NVIDIA GPU detected. Installing CUDA-12 FAISS wheel...")
    p = run([
        sys.executable, "-m", "pip", "install", "-q",
        "faiss-gpu-cu12==1.14.1.post1",
        *common,
    ])
    print(p.stdout[-2000:])

    if p.returncode != 0:
        print("GPU FAISS install failed; falling back to faiss-cpu.")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "faiss-cpu",
            *common,
        ])
else:
    print("No NVIDIA GPU detected; installing faiss-cpu.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "faiss-cpu",
        *common,
    ])

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import gc, hashlib, json, math, random, shutil, time, warnings, zipfile

import faiss
import numpy as np
import pandas as pd
import psutil
import requests
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from google.colab import drive

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20260828
V027_SPLIT_SEED = 20260827

DATASET_NAME = "BEIR Natural Questions"
ENCODER_NAME = "thenlper/gte-small"
DIM = 384
QUERY_PREFIX = ""
PASSAGE_PREFIX = ""

NLIST = 4096
PQ_M = 32
PQ_NBITS = 8

REP_NPROBE = 64
MATCHED_NPROBE = 2
SEARCH_HIGH_NPROBE = 64

TOP_RETRIEVE = 100
UTILITY_K = 10

SHORT_MAX_ROUNDS = 8
LONG_MAX_ROUNDS = 50

SHORT_HORIZONS = [4, 8]
LONG_HORIZONS = [4, 8, 12, 16, 24, 32, 40, 50]
OPERATORS = ["anchored", "recursive"]

ALPHAS = [0.1, 0.3, 0.5, 0.7]
MEAN_K = [5, 20, 50]
SOFTMAX_K = [5, 20]
TEMPERATURES = [0.05, 0.1, 0.2, 0.5]

BOOTSTRAP_REPS = 10_000
EPS_PRIMARY = 0.002

CHECKPOINT_EVERY_QUERIES = 32

ENCODE_BATCH = 1024
CORPUS_BLOCK_ROWS = 100_000
TRAIN_SAMPLE = 500_000

LOCALIZE_CORPUS_BLOCKS = True
LOCAL_FAST_ROOT = Path("/content/arc-v028-nq-gte-fast-cache")

BACKEND_AUDIT_N = 256
BACKEND_MEAN_NDCG_TOL = 5e-4
BACKEND_MEAN_GAP_TOL = 5e-4

EXPECTED_N = 3452
EXPECTED_FIT = 1726
EXPECTED_VAL = 1726

EXPECTED_FIT_SHA = (
    "7b48ad8860b9ea5d639d53126ae48fbe"
    "4c98146658b0ec6fb7facc604e11a201"
)
EXPECTED_VAL_SHA = (
    "8548189d7a7e5fda6150642f5e6eed42"
    "15baa04d4baf389c99acf632ae6e04cf"
)

EXPECTED_V027_CONFIRM_PROTOCOL_SHA = (
    "44a15f18297dce9d5db2b590f4038597"
    "63f916ed0f3cfb3a196d3b2198ecc94a"
)

EXPECTED_V027_REP_GAP = 0.1611140031278725
EXPECTED_V027_SEARCH_GAP = 0.1649720924272112
EXPECTED_V027_REL_MISMATCH = 0.023946331320913332

random.seed(SEED)
np.random.seed(SEED)

if not gpu_present:
    faiss.omp_set_num_threads(os.cpu_count() or 1)

print("faiss:", getattr(faiss, "__version__", "unknown"))
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("CPU threads:", os.cpu_count())
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))

In [ ]:
# Paths: persistent large cache so a disconnected runtime does not force another rebuild.
DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.is_dir():
    drive.mount('/content/drive')

ARC_ROOT = DRIVE_ROOT / 'rag-pq-checkpoints' / 'arc-v0'
V028_ROOT = ARC_ROOT / 'nq-gte-operator-horizon-generalization-v028'
V028_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
OUT = V028_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

LOCAL_RAW_ROOT = Path('/content/arc-v028-nq-raw')
LOCAL_RAW_ROOT.mkdir(parents=True, exist_ok=True)

# Use a Drive-backed cache. It is compatible with the v0.27 NQ-GTE layout.
LARGE_ROOT = DRIVE_ROOT / 'rag-pq-checkpoints' / 'arc-v027-nq-gte-large-cache'
LARGE_ROOT.mkdir(parents=True, exist_ok=True)

print('OUT:', OUT)
print('LARGE_ROOT:', LARGE_ROOT)

In [ ]:
def sha256_file(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def membership_sha(ids):
    return hashlib.sha256('\n'.join(sorted(map(str, ids))).encode('utf-8')).hexdigest()

def iter_jsonl(path):
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def norm_vec(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    return x / max(float(np.linalg.norm(x)), eps)

def slope(y):
    y = np.asarray(y, dtype=np.float64)
    x = np.arange(len(y), dtype=np.float64)
    return float(np.polyfit(x, y, 1)[0])

def jacdist(a, b):
    A = set(map(int, a))
    B = set(map(int, b))
    return 1.0 - len(A & B) / max(1, len(A | B))

def ndcg(ids, relevant, k=10):
    ids = np.asarray(ids, dtype=np.int64)[:k]
    gains = np.asarray([1.0 if int(i) in relevant else 0.0 for i in ids], dtype=np.float64)
    discounts = 1.0 / np.log2(np.arange(2, len(ids) + 2))
    dcg = float(np.sum(gains * discounts))
    m = min(k, len(relevant))
    if m == 0:
        return 0.0
    idcg = float(np.sum(1.0 / np.log2(np.arange(2, m + 2))))
    return dcg / idcg

def mrr(ids, relevant, k=10):
    for rank, d in enumerate(np.asarray(ids)[:k], 1):
        if int(d) in relevant:
            return 1.0 / rank
    return 0.0

def recall_at(ids, relevant, k=10):
    if not relevant:
        return 0.0
    return len(set(map(int, np.asarray(ids)[:k])) & set(relevant)) / len(relevant)

print('Helpers ready.')

## Phase A — Restore BEIR NQ and exact v0.27 split lineage

In [ ]:
NQ_DIR = LOCAL_RAW_ROOT / 'nq'
CORPUS_JSONL = NQ_DIR / 'corpus.jsonl'
QUERIES_JSONL = NQ_DIR / 'queries.jsonl'
ZIP_PATH = LOCAL_RAW_ROOT / 'nq.zip'
NQ_URL = 'https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nq.zip'

def valid_zip(path):
    path = Path(path)
    if not path.is_file() or not zipfile.is_zipfile(path):
        return False
    try:
        with zipfile.ZipFile(path, 'r') as zf:
            return zf.testzip() is None
    except Exception:
        return False

def download_zip_atomic(url, dst):
    dst = Path(dst)
    tmp = dst.with_suffix(dst.suffix + '.part')
    tmp.unlink(missing_ok=True)
    with requests.get(url, stream=True, timeout=(30, 300), allow_redirects=True) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(tmp, 'wb') as f, tqdm(total=total or None, unit='B', unit_scale=True, desc='nq.zip') as bar:
            for chunk in r.iter_content(8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))
    if not valid_zip(tmp):
        prefix = tmp.read_bytes()[:200]
        tmp.unlink(missing_ok=True)
        raise RuntimeError(f'Downloaded NQ artifact is not a valid ZIP. Prefix={prefix!r}')
    tmp.replace(dst)

if not (CORPUS_JSONL.is_file() and QUERIES_JSONL.is_file()):
    if ZIP_PATH.exists() and not valid_zip(ZIP_PATH):
        ZIP_PATH.unlink()
    if not ZIP_PATH.is_file():
        download_zip_atomic(NQ_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(LOCAL_RAW_ROOT)

assert CORPUS_JSONL.is_file()
assert QUERIES_JSONL.is_file()
qrel_candidates = sorted((NQ_DIR / 'qrels').glob('*.tsv'))
assert qrel_candidates
QRELS_PATH = next((p for p in qrel_candidates if p.name == 'test.tsv'), qrel_candidates[0])
print('NQ ready:', CORPUS_JSONL, QUERIES_JSONL, QRELS_PATH, sep='\n')

In [ ]:
qrels_df = pd.read_csv(QRELS_PATH, sep='\t')
q_qid_col = next(c for c in ['query-id', 'query_id', 'qid'] if c in qrels_df.columns)
q_doc_col = next(c for c in ['corpus-id', 'corpus_id', 'doc_id'] if c in qrels_df.columns)
q_score_col = next((c for c in ['score', 'relevance', 'rel'] if c in qrels_df.columns), None)
qrels_df[q_qid_col] = qrels_df[q_qid_col].astype(str)
qrels_df[q_doc_col] = qrels_df[q_doc_col].astype(str)
if q_score_col is not None:
    qrels_df[q_score_col] = pd.to_numeric(qrels_df[q_score_col], errors='coerce')
    qrels_df = qrels_df[qrels_df[q_score_col] > 0].copy()

query_text = {str(o['_id']): str(o.get('text', '')) for o in iter_jsonl(QUERIES_JSONL)}
ALL_QUERY_IDS = sorted(set(qrels_df[q_qid_col]) & set(query_text))
assert len(ALL_QUERY_IDS) == EXPECTED_N

def split_key(qid):
    return hashlib.sha256(f'{V027_SPLIT_SEED}|{qid}'.encode('utf-8')).hexdigest()

ordered = sorted(ALL_QUERY_IDS, key=split_key)
cut = len(ordered) // 2
FIT_IDS = ordered[:cut]
VAL_IDS = ordered[cut:]

assert len(FIT_IDS) == EXPECTED_FIT
assert len(VAL_IDS) == EXPECTED_VAL
assert set(FIT_IDS).isdisjoint(VAL_IDS)
assert membership_sha(FIT_IDS) == EXPECTED_FIT_SHA
assert membership_sha(VAL_IDS) == EXPECTED_VAL_SHA

SPLIT_PATH = OUT / 'v028_reused_v027_query_split.csv'
pd.DataFrame({
    'query_id': FIT_IDS + VAL_IDS,
    'split': ['fit'] * len(FIT_IDS) + ['validation'] * len(VAL_IDS),
}).to_csv(SPLIT_PATH, index=False)

print('V0.27 SPLIT LINEAGE — PASS')
print('FIT:', len(FIT_IDS), EXPECTED_FIT_SHA)
print('VAL:', len(VAL_IDS), EXPECTED_VAL_SHA)

In [ ]:
# Cell 7 — Freeze v0.28 H=50 design before any new validation trajectory.
POLICIES = []

for alpha in ALPHAS:
    for k in MEAN_K:
        POLICIES.append({
            "method": "mean",
            "alpha": float(alpha),
            "k": int(k),
            "temperature": np.nan,
            "config_key": (
                f"mean-k{k}-a{str(alpha).replace('.', 'p')}-tnone"
            ),
        })

    for k in SOFTMAX_K:
        for tau in TEMPERATURES:
            POLICIES.append({
                "method": "softmax",
                "alpha": float(alpha),
                "k": int(k),
                "temperature": float(tau),
                "config_key": (
                    f"softmax-k{k}-a{str(alpha).replace('.', 'p')}-"
                    f"t{str(tau).replace('.', 'p')}"
                ),
            })

assert len(POLICIES) == 44

LONG_POLICY_KEYS = []
for alpha in ALPHAS:
    a = str(alpha).replace(".", "p")
    LONG_POLICY_KEYS.extend([
        f"mean-k20-a{a}-tnone",
        f"softmax-k20-a{a}-t0p1",
    ])

LONG_POLICIES = [
    p for p in POLICIES
    if p["config_key"] in LONG_POLICY_KEYS
]
SHORT_ONLY_POLICIES = [
    p for p in POLICIES
    if p["config_key"] not in LONG_POLICY_KEYS
]

assert len(LONG_POLICIES) == 8
assert len(SHORT_ONLY_POLICIES) == 36
assert set(p["config_key"] for p in LONG_POLICIES) == set(LONG_POLICY_KEYS)

POLICY_PATH = OUT / "v028_frozen_policy_grid.csv"
pd.DataFrame(POLICIES).to_csv(POLICY_PATH, index=False)

LONG_POLICY_PATH = OUT / "v028_frozen_h50_policy_subset.csv"
pd.DataFrame(LONG_POLICIES).to_csv(LONG_POLICY_PATH, index=False)

PROTOCOL = {
    "status": (
        "ARC_V028_H50_A100_BATCHED_AGENTIC_BRIDGE_"
        "FROZEN_BEFORE_NEW_VALIDATION_TRAJECTORIES"
    ),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "classification": (
        "post-v0.27 reviewer-oriented prospective operator/horizon audit; "
        "controlled bridge toward Agentic IR/RAG, not a full LLM-agent experiment"
    ),
    "dataset": DATASET_NAME,
    "encoder": ENCODER_NAME,
    "v027_confirmation_protocol_sha256": EXPECTED_V027_CONFIRM_PROTOCOL_SHA,
    "split": {
        "n_fit": len(FIT_IDS),
        "n_validation": len(VAL_IDS),
        "fit_sha256": membership_sha(FIT_IDS),
        "validation_sha256": membership_sha(VAL_IDS),
    },
    "frozen_severity_match_reused_without_retuning": {
        "representation": "IVF-PQ32 -> IVF-SQ8 at nprobe=64",
        "search_effort": "IVF-SQ8 nprobe=2 -> 64",
        "representation_gap_ndcg10": EXPECTED_V027_REP_GAP,
        "search_gap_ndcg10": EXPECTED_V027_SEARCH_GAP,
        "relative_gap_mismatch": EXPECTED_V027_REL_MISMATCH,
        "matched_nprobe_low": MATCHED_NPROBE,
    },
    "operators": {
        "anchored": "q_(t+1)=normalize((1-alpha)q0 + alpha*F_t)",
        "recursive": "q_(t+1)=normalize((1-alpha)q_t + alpha*F_t)",
    },
    "tier_A_full_policy": {
        "policy_count": len(POLICIES),
        "max_feedback_updates": SHORT_MAX_ROUNDS,
        "analysis_horizons": SHORT_HORIZONS,
    },
    "tier_B_h50_agentic_bridge": {
        "policy_count": len(LONG_POLICIES),
        "selection_rule": (
            "for each alpha: mean-k20 and softmax-k20-t0.1; "
            "structural selection frozen before validation outcomes"
        ),
        "config_keys": LONG_POLICY_KEYS,
        "max_feedback_updates": LONG_MAX_ROUNDS,
        "analysis_horizons": LONG_HORIZONS,
    },
    "execution_plan": {
        "preferred_backend": "single-GPU CUDA FAISS",
        "fallback_backend": "batched CPU FAISS",
        "query_batch_size": CHECKPOINT_EVERY_QUERIES,
        "localize_corpus_embedding_blocks": LOCALIZE_CORPUS_BLOCKS,
        "fit_only_cpu_gpu_semantic_audit": True,
    },
    "primary_new_estimand": (
        "recursive H=50 long-policy-subset query-averaged "
        "nDCG@10 H3abs representation minus severity-matched search effort"
    ),
    "primary_success_criterion": (
        "10,000-replicate paired query-bootstrap 95% CI strictly above zero"
    ),
    "secondary_prespecified": [
        "recursive H12/H16/H24/H32/H40 paired ordering",
        "anchored H12/H16/H24/H32/H40/H50 paired ordering",
        "full-policy anchored/recursive H4/H8 paired ordering",
        "mean-only, softmax-only, equal-family paired ordering",
        "MRR@10 and Recall@10 H3abs paired ordering",
        "H50-minus-H8 persistence/saturation/late-reversal diagnostics",
        "raw round50-minus-round8 absolute-gap growth",
        "peak-gap round distribution",
    ],
    "validation_retuning_allowed": False,
    "negative_null_saturation_reversal_retained": True,
}

PROTOCOL_PATH = OUT / "v028_frozen_h50_agentic_bridge_protocol.json"

PROTOCOL_PATH.write_text(
    json.dumps(PROTOCOL, indent=2, sort_keys=True),
    encoding="utf-8",
)

PROTOCOL_SHA = sha256_file(PROTOCOL_PATH)

(OUT / "V028_PROTOCOL_SHA256.txt").write_text(
    f"{PROTOCOL_SHA}  {PROTOCOL_PATH.name}\n",
    encoding="utf-8",
)

print("V0.28 H=50 PROTOCOL FROZEN — PASS")
print("Protocol SHA:", PROTOCOL_SHA)
print("Full policies:", len(POLICIES))
print("H=50 policies:", len(LONG_POLICIES))
print("Long horizons:", LONG_HORIZONS)

## Phase B — Restore/build NQ-GTE embeddings and frozen IVF indexes

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer(ENCODER_NAME, device=device)

QUERY_IDS_PATH = LARGE_ROOT / 'nq_query_ids.txt'
QUERY_EMB_PATH = LARGE_ROOT / 'nq_query_embeddings.float32.npy'

if QUERY_EMB_PATH.is_file() and QUERY_IDS_PATH.is_file():
    query_embeddings = np.load(QUERY_EMB_PATH, mmap_mode='r')
    assert QUERY_IDS_PATH.read_text(encoding='utf-8').splitlines() == ALL_QUERY_IDS
    print('Reusing query embeddings.')
else:
    query_embeddings = model.encode(
        [QUERY_PREFIX + query_text[qid] for qid in ALL_QUERY_IDS],
        batch_size=ENCODE_BATCH,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)
    np.save(QUERY_EMB_PATH, query_embeddings)
    QUERY_IDS_PATH.write_text('\n'.join(ALL_QUERY_IDS) + '\n', encoding='utf-8')

QUERY_INDEX = {qid: i for i, qid in enumerate(ALL_QUERY_IDS)}
assert query_embeddings.shape == (EXPECTED_N, DIM)
print('Query embeddings:', query_embeddings.shape, 'device:', device)

In [ ]:
BLOCK_DIR = LARGE_ROOT / 'corpus_blocks'
BLOCK_DIR.mkdir(parents=True, exist_ok=True)
QREL_ROW_MAP_PATH = LARGE_ROOT / 'nq_qrel_doc_rows.csv'
CORPUS_MANIFEST_PATH = LARGE_ROOT / 'nq_corpus_block_manifest.json'
RELEVANT_DOC_IDS = set(qrels_df[q_doc_col].astype(str))

block_records = []
relevant_row_map = {}
texts_buf = []
block_idx = 0
row_count = 0

def save_or_reuse_block(idx, texts, row_start):
    ep = BLOCK_DIR / f'block-{idx:04d}.float16.npy'
    if ep.is_file():
        arr = np.load(ep, mmap_mode='r')
        assert arr.shape == (len(texts), DIM), (ep, arr.shape, len(texts))
        return {
            'block': idx, 'row_start': row_start, 'row_end': row_start + len(texts),
            'rows': len(texts), 'path': str(ep), 'sha256': sha256_file(ep), 'reused': True,
        }
    vec = model.encode(
        [PASSAGE_PREFIX + t for t in texts],
        batch_size=ENCODE_BATCH,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float16)
    np.save(ep, vec)
    return {
        'block': idx, 'row_start': row_start, 'row_end': row_start + len(texts),
        'rows': len(texts), 'path': str(ep), 'sha256': sha256_file(ep), 'reused': False,
    }

for obj in iter_jsonl(CORPUS_JSONL):
    doc_id = str(obj['_id'])
    title = str(obj.get('title', ''))
    text = str(obj.get('text', ''))
    full = (title + ' ' + text).strip() if title else text
    if doc_id in RELEVANT_DOC_IDS:
        relevant_row_map[doc_id] = row_count
    texts_buf.append(full)
    row_count += 1
    if len(texts_buf) >= CORPUS_BLOCK_ROWS:
        start = row_count - len(texts_buf)
        block_records.append(save_or_reuse_block(block_idx, texts_buf, start))
        texts_buf = []
        block_idx += 1

if texts_buf:
    start = row_count - len(texts_buf)
    block_records.append(save_or_reuse_block(block_idx, texts_buf, start))

N_DOCS = row_count
assert N_DOCS == 2_681_468, N_DOCS
assert sum(r['rows'] for r in block_records) == N_DOCS
missing = RELEVANT_DOC_IDS - set(relevant_row_map)
assert not missing, list(missing)[:10]

pd.DataFrame([
    {'doc_id': d, 'corpus_row': r} for d, r in relevant_row_map.items()
]).to_csv(QREL_ROW_MAP_PATH, index=False)

CORPUS_MANIFEST_PATH.write_text(json.dumps({
    'status': 'COMPLETE', 'encoder': ENCODER_NAME, 'dimension': DIM,
    'corpus_rows': N_DOCS, 'blocks': block_records,
    'qrels_relevant_docs_mapped': len(relevant_row_map),
}, indent=2), encoding='utf-8')

print('Corpus rows:', f'{N_DOCS:,}', 'blocks:', len(block_records))

In [ ]:
# Cell 11 — Localize corpus blocks to local SSD and build vectorized row fetch.
blocks = []
block_starts = []
block_ends = []

if LOCALIZE_CORPUS_BLOCKS:
    LOCAL_FAST_ROOT.mkdir(parents=True, exist_ok=True)
    free_gib = shutil.disk_usage("/content").free / 2**30
    source_bytes = sum(Path(r["path"]).stat().st_size for r in block_records)
    source_gib = source_bytes / 2**30

    print("Corpus block size GiB:", round(source_gib, 3))
    print("Local free GiB:", round(free_gib, 3))

    if free_gib < source_gib + 5:
        print("Insufficient local SSD headroom; using Drive-backed blocks.")
        LOCALIZE_CORPUS_BLOCKS = False

for r in tqdm(block_records, desc="Preparing corpus block memmaps"):
    src_path = Path(r["path"])
    use_path = src_path

    if LOCALIZE_CORPUS_BLOCKS:
        dst = LOCAL_FAST_ROOT / src_path.name
        if not dst.is_file() or dst.stat().st_size != src_path.stat().st_size:
            tmp = dst.with_suffix(dst.suffix + ".tmp")
            shutil.copy2(src_path, tmp)
            os.replace(tmp, dst)
        use_path = dst

    arr = np.load(use_path, mmap_mode="r")
    assert arr.shape == (int(r["rows"]), DIM)

    blocks.append(arr)
    block_starts.append(int(r["row_start"]))
    block_ends.append(int(r["row_end"]))

block_starts = np.asarray(block_starts, dtype=np.int64)
block_ends = np.asarray(block_ends, dtype=np.int64)

def fetch_docs(rows):
    rows = np.asarray(rows, dtype=np.int64)
    original_shape = rows.shape
    flat = rows.reshape(-1)

    out = np.empty((len(flat), DIM), dtype=np.float32)
    bidx = np.searchsorted(block_ends, flat, side="right")

    if (bidx >= len(blocks)).any():
        raise IndexError("Corpus row outside block range.")

    for b in np.unique(bidx):
        mask = bidx == b
        local = flat[mask] - block_starts[b]
        out[mask] = np.asarray(blocks[int(b)][local], dtype=np.float32)

    out /= np.maximum(
        np.linalg.norm(out, axis=1, keepdims=True),
        1e-12,
    )

    return out.reshape(*original_shape, DIM)

row_map_df = pd.read_csv(
    QREL_ROW_MAP_PATH,
    dtype={"doc_id": str},
)

DOC_TO_ROW = dict(
    zip(
        row_map_df["doc_id"],
        row_map_df["corpus_row"].astype(int),
    )
)

QRELS = defaultdict(set)

for _, r in qrels_df.iterrows():
    QRELS[str(r[q_qid_col])].add(
        int(DOC_TO_ROW[str(r[q_doc_col])])
    )

assert all(q in QRELS and QRELS[q] for q in ALL_QUERY_IDS)

smoke_rows = np.array(list(DOC_TO_ROW.values())[:256], dtype=np.int64)
smoke_vec = fetch_docs(smoke_rows)

assert smoke_vec.shape == (len(smoke_rows), DIM)
assert np.isfinite(smoke_vec).all()

print("Qrels row-space alignment — PASS")
print(
    "Corpus block backend:",
    "LOCAL_SSD" if LOCALIZE_CORPUS_BLOCKS else "DRIVE_MEMMAP",
)

In [ ]:
# Cell 12 — Restore/build CPU indexes, then create audited GPU execution indexes.
TRAIN_PATH = LARGE_ROOT / f"train_sample_{TRAIN_SAMPLE}.float32.npy"

if TRAIN_PATH.is_file():
    train = np.load(TRAIN_PATH, mmap_mode="r")
    assert train.shape == (TRAIN_SAMPLE, DIM)
    print("Reusing train sample.")
else:
    rng = np.random.default_rng(V027_SPLIT_SEED)
    sample_rows = np.sort(
        rng.choice(N_DOCS, size=TRAIN_SAMPLE, replace=False)
    )

    train = np.empty((TRAIN_SAMPLE, DIM), dtype=np.float32)

    chunk = 20_000
    for s in tqdm(
        range(0, TRAIN_SAMPLE, chunk),
        desc="Building index train sample",
    ):
        e = min(s + chunk, TRAIN_SAMPLE)
        train[s:e] = fetch_docs(sample_rows[s:e])

    np.save(TRAIN_PATH, train)

PQ_PATH = LARGE_ROOT / "nq-gte-ivfpq-nlist4096-m32-nbits8.faiss"
SQ_PATH = LARGE_ROOT / "nq-gte-ivfsq8-nlist4096.faiss"

if PQ_PATH.is_file() and SQ_PATH.is_file():
    print("Loading CPU canonical indexes...")
    pq32 = faiss.read_index(str(PQ_PATH))
    sq8 = faiss.read_index(str(SQ_PATH))
else:
    print("Building canonical CPU IVF-PQ32 and IVF-SQ8 indexes...")

    quant_pq = faiss.IndexFlatIP(DIM)
    pq32 = faiss.IndexIVFPQ(
        quant_pq,
        DIM,
        NLIST,
        PQ_M,
        PQ_NBITS,
        faiss.METRIC_INNER_PRODUCT,
    )
    pq32.train(np.asarray(train, dtype=np.float32))

    quant_sq = faiss.IndexFlatIP(DIM)
    sq8 = faiss.IndexIVFScalarQuantizer(
        quant_sq,
        DIM,
        NLIST,
        faiss.ScalarQuantizer.QT_8bit,
        faiss.METRIC_INNER_PRODUCT,
    )
    sq8.train(np.asarray(train, dtype=np.float32))

    for arr in tqdm(blocks, desc="Adding corpus blocks"):
        x = np.asarray(arr, dtype=np.float32)
        x /= np.maximum(
            np.linalg.norm(x, axis=1, keepdims=True),
            1e-12,
        )
        pq32.add(x)
        sq8.add(x)

    faiss.write_index(pq32, str(PQ_PATH))
    faiss.write_index(sq8, str(SQ_PATH))

assert pq32.ntotal == N_DOCS
assert sq8.ntotal == N_DOCS

pq_exec = pq32
sq_exec = sq8
EXECUTION_BACKEND = "CPU_BATCHED"
GPU_RESOURCES = None

if torch.cuda.is_available() and hasattr(faiss, "StandardGpuResources"):
    print("Attempting to clone both IVF indexes to GPU...")

    try:
        GPU_RESOURCES = faiss.StandardGpuResources()

        try:
            GPU_RESOURCES.setTempMemory(2 * 1024**3)
        except Exception:
            pass

        opts = faiss.GpuClonerOptions()
        opts.useFloat16 = False

        pq_gpu = faiss.index_cpu_to_gpu(
            GPU_RESOURCES,
            0,
            pq32,
            opts,
        )
        sq_gpu = faiss.index_cpu_to_gpu(
            GPU_RESOURCES,
            0,
            sq8,
            opts,
        )

        assert pq_gpu.ntotal == N_DOCS
        assert sq_gpu.ntotal == N_DOCS

        pq_exec = pq_gpu
        sq_exec = sq_gpu
        EXECUTION_BACKEND = "GPU_BATCHED"

        print("GPU clone — PASS")

    except Exception as e:
        print("GPU clone failed; using CPU batched backend.")
        print("Reason:", repr(e))

        pq_exec = pq32
        sq_exec = sq8
        EXECUTION_BACKEND = "CPU_BATCHED"

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Canonical PQ:", PQ_PATH)
print("Canonical SQ:", SQ_PATH)
print("Execution backend:", EXECUTION_BACKEND)

## Phase C — FIT-only implementation smoke, then untouched validation sweep

In [ ]:
# Cell 14 — Batched search / feedback / trajectory implementation.
def norm_rows(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)

    return x / np.maximum(
        np.linalg.norm(x, axis=1, keepdims=True),
        eps,
    )

def search_index_batch(index, Q, nprobe):
    index.nprobe = int(nprobe)
    Q = norm_rows(Q)

    scores, ids = index.search(
        Q,
        TOP_RETRIEVE,
    )

    if (ids < 0).any():
        raise RuntimeError("FAISS returned invalid IDs.")

    return np.asarray(scores), np.asarray(ids)

def feedback_vector_batch(scores, ids, policy):
    k = int(policy["k"])

    ids_k = np.asarray(
        ids[:, :k],
        dtype=np.int64,
    )
    scores_k = np.asarray(
        scores[:, :k],
        dtype=np.float64,
    )

    docs = fetch_docs(ids_k)

    if policy["method"] == "mean":
        f = docs.mean(axis=1)
    else:
        tau = float(policy["temperature"])
        z = scores_k / tau
        z -= z.max(axis=1, keepdims=True)

        w = np.exp(z)
        w /= w.sum(axis=1, keepdims=True)

        f = (docs * w[:, :, None]).sum(axis=1)

    return norm_rows(f)

def update_state_batch(
    Q0,
    Qcurrent,
    feedback,
    alpha,
    operator,
):
    alpha = float(alpha)
    base = Q0 if operator == "anchored" else Qcurrent

    return norm_rows(
        (1.0 - alpha) * base
        + alpha * feedback
    )

def metric_triplet(ids, rel):
    return {
        "ndcg10": ndcg(ids, rel, 10),
        "mrr10": mrr(ids, rel, 10),
        "recall10": recall_at(ids, rel, 10),
    }

def candidate_jaccard_batch(a, b):
    return np.asarray(
        [
            jacdist(
                x[:TOP_RETRIEVE],
                y[:TOP_RETRIEVE],
            )
            for x, y in zip(a, b)
        ],
        dtype=np.float64,
    )

def run_operator_triplet_batch(
    qids,
    policy,
    operator,
    max_rounds,
):
    qids = list(map(str, qids))
    B = len(qids)

    idx = [QUERY_INDEX[q] for q in qids]

    Q0 = norm_rows(
        np.asarray(
            query_embeddings[idx],
            dtype=np.float32,
        )
    )

    QP = Q0.copy()
    QN = Q0.copy()
    QH = Q0.copy()

    rels = [QRELS[q] for q in qids]
    records = []

    tier = (
        "long_horizon"
        if policy["config_key"] in LONG_POLICY_KEYS
        else "short_full_policy"
    )

    for t in range(max_rounds + 1):
        sP, iP = search_index_batch(
            pq_exec,
            QP,
            REP_NPROBE,
        )
        sN, iN = search_index_batch(
            sq_exec,
            QN,
            MATCHED_NPROBE,
        )
        sH, iH = search_index_batch(
            sq_exec,
            QH,
            SEARCH_HIGH_NPROBE,
        )

        state_P = 1.0 - np.sum(
            norm_rows(QP) * norm_rows(QH),
            axis=1,
        )
        state_N = 1.0 - np.sum(
            norm_rows(QN) * norm_rows(QH),
            axis=1,
        )

        jac_P = candidate_jaccard_batch(iP, iH)
        jac_N = candidate_jaccard_batch(iN, iH)

        for j, qid in enumerate(qids):
            mP = metric_triplet(iP[j], rels[j])
            mN = metric_triplet(iN[j], rels[j])
            mH = metric_triplet(iH[j], rels[j])

            common = {
                "query_id": qid,
                "operator": operator,
                "iteration": int(t),
                "method": policy["method"],
                "alpha": float(policy["alpha"]),
                "k": int(policy["k"]),
                "temperature": (
                    float(policy["temperature"])
                    if not pd.isna(policy["temperature"])
                    else np.nan
                ),
                "config_key": policy["config_key"],
                "trajectory_tier": tier,
                "execution_backend": EXECUTION_BACKEND,
            }

            for mech, state, jac, mL in [
                ("representation", state_P[j], jac_P[j], mP),
                ("search_effort", state_N[j], jac_N[j], mN),
            ]:
                row = dict(common)

                row.update({
                    "mechanism": mech,
                    "query_state_distance": float(state),
                    "candidate_jaccard_distance": float(jac),
                })

                for metric in [
                    "ndcg10",
                    "mrr10",
                    "recall10",
                ]:
                    low = float(mL[metric])
                    high = float(mH[metric])

                    row[f"{metric}_low"] = low
                    row[f"{metric}_high"] = high
                    row[f"{metric}_signed_gap"] = high - low
                    row[f"{metric}_abs_gap"] = abs(high - low)

                records.append(row)

        if t == max_rounds:
            break

        fP = feedback_vector_batch(
            sP,
            iP,
            policy,
        )
        fN = feedback_vector_batch(
            sN,
            iN,
            policy,
        )
        fH = feedback_vector_batch(
            sH,
            iH,
            policy,
        )

        QP = update_state_batch(
            Q0,
            QP,
            fP,
            policy["alpha"],
            operator,
        )
        QN = update_state_batch(
            Q0,
            QN,
            fN,
            policy["alpha"],
            operator,
        )
        QH = update_state_batch(
            Q0,
            QH,
            fH,
            policy["alpha"],
            operator,
        )

    return pd.DataFrame.from_records(records)

print("Batched trajectory implementation ready.")
print("Execution backend:", EXECUTION_BACKEND)

In [ ]:
# Cell 15 — FIT-only backend semantic audit + H=50 implementation smoke.
# Validation trajectories remain untouched.

AUDIT_IDS = FIT_IDS[: min(BACKEND_AUDIT_N, len(FIT_IDS))]
audit_idx = [QUERY_INDEX[q] for q in AUDIT_IDS]

QA = norm_rows(
    np.asarray(
        query_embeddings[audit_idx],
        dtype=np.float32,
    )
)

def batch_metrics_for_backend(pq_index, sq_index):
    _, iP = search_index_batch(
        pq_index,
        QA,
        REP_NPROBE,
    )
    _, iN = search_index_batch(
        sq_index,
        QA,
        MATCHED_NPROBE,
    )
    _, iH = search_index_batch(
        sq_index,
        QA,
        SEARCH_HIGH_NPROBE,
    )

    p = np.asarray([
        ndcg(i, QRELS[q], 10)
        for i, q in zip(iP, AUDIT_IDS)
    ])
    n = np.asarray([
        ndcg(i, QRELS[q], 10)
        for i, q in zip(iN, AUDIT_IDS)
    ])
    h = np.asarray([
        ndcg(i, QRELS[q], 10)
        for i, q in zip(iH, AUDIT_IDS)
    ])

    return p, n, h

cpu_p, cpu_n, cpu_h = batch_metrics_for_backend(
    pq32,
    sq8,
)

if EXECUTION_BACKEND == "GPU_BATCHED":
    gpu_p, gpu_n, gpu_h = batch_metrics_for_backend(
        pq_exec,
        sq_exec,
    )

    audit = {
        "n_fit_queries": len(AUDIT_IDS),
        "max_abs_ndcg_diff_representation_low": float(
            np.max(np.abs(gpu_p - cpu_p))
        ),
        "max_abs_ndcg_diff_search_low": float(
            np.max(np.abs(gpu_n - cpu_n))
        ),
        "max_abs_ndcg_diff_high": float(
            np.max(np.abs(gpu_h - cpu_h))
        ),
        "mean_ndcg_diff_representation_low": float(
            np.mean(gpu_p - cpu_p)
        ),
        "mean_ndcg_diff_search_low": float(
            np.mean(gpu_n - cpu_n)
        ),
        "mean_ndcg_diff_high": float(
            np.mean(gpu_h - cpu_h)
        ),
        "cpu_rep_gap": float(np.mean(cpu_h - cpu_p)),
        "gpu_rep_gap": float(np.mean(gpu_h - gpu_p)),
        "cpu_search_gap": float(np.mean(cpu_h - cpu_n)),
        "gpu_search_gap": float(np.mean(gpu_h - gpu_n)),
    }

    audit["rep_gap_abs_backend_delta"] = abs(
        audit["gpu_rep_gap"]
        - audit["cpu_rep_gap"]
    )
    audit["search_gap_abs_backend_delta"] = abs(
        audit["gpu_search_gap"]
        - audit["cpu_search_gap"]
    )

    mean_metric_delta = max(
        abs(audit["mean_ndcg_diff_representation_low"]),
        abs(audit["mean_ndcg_diff_search_low"]),
        abs(audit["mean_ndcg_diff_high"]),
    )

    backend_pass = bool(
        mean_metric_delta <= BACKEND_MEAN_NDCG_TOL
        and audit["rep_gap_abs_backend_delta"] <= BACKEND_MEAN_GAP_TOL
        and audit["search_gap_abs_backend_delta"] <= BACKEND_MEAN_GAP_TOL
    )

    print(json.dumps(audit, indent=2))
    print("GPU semantic audit PASS:", backend_pass)

    if not backend_pass:
        print(
            "GPU backend differs beyond frozen tolerance; "
            "falling back to CPU_BATCHED before validation."
        )

        pq_exec = pq32
        sq_exec = sq8
        EXECUTION_BACKEND = "CPU_BATCHED"

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

else:
    audit = {
        "n_fit_queries": len(AUDIT_IDS),
        "execution_backend": EXECUTION_BACKEND,
        "gpu_semantic_audit": "not applicable",
    }

AUDIT_PATH = OUT / "v028_fit_execution_backend_audit.json"

AUDIT_PATH.write_text(
    json.dumps(audit, indent=2, sort_keys=True),
    encoding="utf-8",
)

smoke_qids = FIT_IDS[: min(8, len(FIT_IDS))]

p_short = SHORT_ONLY_POLICIES[0]
p_long = LONG_POLICIES[0]

for operator in OPERATORS:
    s = run_operator_triplet_batch(
        smoke_qids,
        p_short,
        operator,
        SHORT_MAX_ROUNDS,
    )

    assert len(s) == (
        len(smoke_qids)
        * 2
        * (SHORT_MAX_ROUNDS + 1)
    )
    assert s["iteration"].max() == SHORT_MAX_ROUNDS

    l = run_operator_triplet_batch(
        smoke_qids,
        p_long,
        operator,
        LONG_MAX_ROUNDS,
    )

    assert len(l) == (
        len(smoke_qids)
        * 2
        * (LONG_MAX_ROUNDS + 1)
    )
    assert l["iteration"].max() == LONG_MAX_ROUNDS

    for col in l.select_dtypes(
        include=[np.number]
    ).columns:
        vals = l[col].dropna().to_numpy()
        assert np.isfinite(vals).all(), col

print("FIT-ONLY H=50 BATCHED SMOKE — PASS")
print("Final execution backend:", EXECUTION_BACKEND)
print("Validation trajectories untouched in this notebook before Cell 16.")

In [ ]:
# Cell 16 — Resumable batched validation sweep.
RUN_DIR = OUT / "validation-h50-agentic-bridge"
RUN_DIR.mkdir(parents=True, exist_ok=True)

short_rows_per_query = (
    len(SHORT_ONLY_POLICIES)
    * len(OPERATORS)
    * 2
    * (SHORT_MAX_ROUNDS + 1)
)

long_rows_per_query = (
    len(LONG_POLICIES)
    * len(OPERATORS)
    * 2
    * (LONG_MAX_ROUNDS + 1)
)

expected_rows = len(VAL_IDS) * (
    short_rows_per_query
    + long_rows_per_query
)

print("VALIDATION QUERIES:", len(VAL_IDS))
print("SHORT-ONLY POLICIES:", len(SHORT_ONLY_POLICIES), "through H=8")
print("H=50 POLICIES:", len(LONG_POLICIES), "through H=50")
print("OPERATORS:", OPERATORS)
print("QUERY BATCH:", CHECKPOINT_EVERY_QUERIES)
print("EXECUTION BACKEND:", EXECUTION_BACKEND)
print("EXPECTED TRAJECTORY ROWS:", f"{expected_rows:,}")
print("CHECKPOINT DIR:", RUN_DIR)

for start in range(
    0,
    len(VAL_IDS),
    CHECKPOINT_EVERY_QUERIES,
):
    stop = min(
        start + CHECKPOINT_EVERY_QUERIES,
        len(VAL_IDS),
    )

    cp = RUN_DIR / f"traj_{start:05d}_{stop:05d}.parquet"

    if cp.exists():
        print("skip", cp.name)
        continue

    qids = VAL_IDS[start:stop]

    frames = []
    t0 = time.perf_counter()

    for policy in SHORT_ONLY_POLICIES:
        for operator in OPERATORS:
            frames.append(
                run_operator_triplet_batch(
                    qids,
                    policy,
                    operator,
                    SHORT_MAX_ROUNDS,
                )
            )

    for policy in LONG_POLICIES:
        for operator in OPERATORS:
            frames.append(
                run_operator_triplet_batch(
                    qids,
                    policy,
                    operator,
                    LONG_MAX_ROUNDS,
                )
            )

    df = pd.concat(frames, ignore_index=True)

    expected_cp = len(qids) * (
        short_rows_per_query
        + long_rows_per_query
    )

    assert len(df) == expected_cp, (
        len(df),
        expected_cp,
    )

    tmp = cp.with_suffix(".tmp.parquet")
    df.to_parquet(tmp, index=False)
    os.replace(tmp, cp)

    elapsed = time.perf_counter() - t0

    completed = stop
    rate = len(qids) / elapsed
    remaining = len(VAL_IDS) - completed
    eta_sec = remaining / max(rate, 1e-12)

    print(
        f"wrote {cp.name}: {len(df):,} rows | "
        f"{elapsed:.1f}s | "
        f"{rate:.3f} q/s | "
        f"rough ETA {eta_sec/3600:.2f} h"
    )

parts = sorted(
    RUN_DIR.glob("traj_*.parquet")
)
assert parts

actual_rows = 0
checkpoint_rows = []

for p in tqdm(
    parts,
    desc="Auditing checkpoint row counts",
):
    pf = pd.read_parquet(
        p,
        columns=["query_id"],
    )

    n = len(pf)
    actual_rows += n

    checkpoint_rows.append({
        "file": p.name,
        "bytes": p.stat().st_size,
        "rows": n,
        "sha256": sha256_file(p),
    })

assert actual_rows == expected_rows, (
    actual_rows,
    expected_rows,
)

CHECKPOINT_MANIFEST_PATH = (
    OUT / "v028_validation_checkpoint_manifest.csv"
)

pd.DataFrame(
    checkpoint_rows
).to_csv(
    CHECKPOINT_MANIFEST_PATH,
    index=False,
)

print("VALIDATION SWEEP — COMPLETE")
print("Trajectory rows:", f"{actual_rows:,}")
print("Checkpoint count:", len(parts))

## Phase D — Frozen primary analysis and pre-specified secondary robustness

In [ ]:
# Cell 18 — Streaming endpoint derivation from trajectory checkpoints.
def endpoint_df_by_horizon(traj):
    out = []

    group_cols = [
        "query_id",
        "operator",
        "mechanism",
        "method",
        "alpha",
        "k",
        "temperature",
        "config_key",
        "trajectory_tier",
    ]

    for keys, g in traj.groupby(
        group_cols,
        dropna=False,
        sort=False,
    ):
        g = g.sort_values("iteration")
        max_available = int(g["iteration"].max())

        horizons = [
            h
            for h in LONG_HORIZONS
            if h <= max_available
        ]

        for horizon in horizons:
            h = g[
                g["iteration"] <= horizon
            ]

            assert len(h) == horizon + 1

            rec = {
                "query_id": keys[0],
                "operator": keys[1],
                "mechanism": keys[2],
                "method": keys[3],
                "alpha": keys[4],
                "k": keys[5],
                "temperature": keys[6],
                "config_key": keys[7],
                "trajectory_tier": keys[8],
                "horizon": horizon,
                "H1_slope": slope(
                    h["query_state_distance"]
                ),
                "H2_slope": slope(
                    h["candidate_jaccard_distance"]
                ),
            }

            for metric in [
                "ndcg10",
                "mrr10",
                "recall10",
            ]:
                abs_col = f"{metric}_abs_gap"
                signed_col = f"{metric}_signed_gap"

                rec[f"{metric}_H3_abs_slope"] = slope(
                    h[abs_col]
                )
                rec[f"{metric}_H3_signed_slope"] = slope(
                    h[signed_col]
                )
                rec[
                    f"{metric}_final_minus_initial_abs_gap"
                ] = float(
                    h[abs_col].iloc[-1]
                    - h[abs_col].iloc[0]
                )
                rec[f"{metric}_final_abs_gap"] = float(
                    h[abs_col].iloc[-1]
                )
                rec[f"{metric}_peak_abs_gap"] = float(
                    h[abs_col].max()
                )
                rec[f"{metric}_peak_round"] = int(
                    h.loc[
                        h[abs_col].idxmax(),
                        "iteration",
                    ]
                )

            out.append(rec)

    return pd.DataFrame(out)

ENDPOINT_DIR = OUT / "endpoint-checkpoints"
ENDPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

endpoint_parts = []

for tp in tqdm(
    parts,
    desc="Computing endpoint checkpoints",
):
    ep = ENDPOINT_DIR / tp.name.replace(
        "traj_",
        "endpoint_",
    )

    if not ep.exists():
        tdf = pd.read_parquet(tp)
        edf = endpoint_df_by_horizon(tdf)

        tmp = ep.with_suffix(".tmp.parquet")
        edf.to_parquet(tmp, index=False)
        os.replace(tmp, ep)

        del tdf, edf
        gc.collect()

    endpoint_parts.append(ep)

endpoints = pd.concat(
    [
        pd.read_parquet(p)
        for p in endpoint_parts
    ],
    ignore_index=True,
)

ENDPOINT_PATH = (
    OUT / "v028_h50_agentic_bridge_endpoints.parquet"
)

endpoints.to_parquet(
    ENDPOINT_PATH,
    index=False,
)

print("Endpoint rows:", f"{len(endpoints):,}")
print(
    endpoints.groupby(
        ["trajectory_tier", "horizon"]
    ).size()
)

In [ ]:
# Cell 19 — Query-level bootstrap: full-policy H4/H8 + long-subset H4...H50.
rng = np.random.default_rng(
    SEED + 2801
)

def boot(x, reps=BOOTSTRAP_REPS):
    x = np.asarray(
        x,
        dtype=np.float64,
    )
    n = len(x)

    b = np.empty(
        reps,
        dtype=np.float64,
    )

    for i in range(reps):
        idx = rng.integers(
            0,
            n,
            size=n,
        )
        b[i] = float(
            x[idx].mean()
        )

    lo, hi = np.quantile(
        b,
        [0.025, 0.975],
    )

    return (
        float(x.mean()),
        float(lo),
        float(hi),
        n,
    )

def query_mechanism_values(
    metric,
    operator,
    horizon,
    long_only=False,
):
    col = f"{metric}_H3_abs_slope"

    x = endpoints[
        (endpoints.operator == operator)
        & (endpoints.horizon == horizon)
    ].copy()

    if long_only:
        x = x[
            x.trajectory_tier
            == "long_horizon"
        ]

    q = x.groupby(
        ["query_id", "mechanism"],
        as_index=False,
    )[col].mean()

    rep = (
        q[
            q.mechanism
            == "representation"
        ]
        .set_index("query_id")
        .loc[VAL_IDS, col]
        .to_numpy(float)
    )

    sea = (
        q[
            q.mechanism
            == "search_effort"
        ]
        .set_index("query_id")
        .loc[VAL_IDS, col]
        .to_numpy(float)
    )

    return rep, sea

rows = []

for metric in [
    "ndcg10",
    "mrr10",
    "recall10",
]:
    for operator in OPERATORS:
        for horizon in SHORT_HORIZONS:
            rep, sea = query_mechanism_values(
                metric,
                operator,
                horizon,
                long_only=False,
            )

            rs = boot(rep)
            ss = boot(sea)
            ds = boot(rep - sea)

            rows.extend([
                {
                    "analysis_tier": "full_policy",
                    "metric": metric,
                    "operator": operator,
                    "horizon": horizon,
                    "estimand": "representation_H3abs",
                    "mean": rs[0],
                    "ci95_low": rs[1],
                    "ci95_high": rs[2],
                    "n_queries": rs[3],
                },
                {
                    "analysis_tier": "full_policy",
                    "metric": metric,
                    "operator": operator,
                    "horizon": horizon,
                    "estimand": "search_H3abs",
                    "mean": ss[0],
                    "ci95_low": ss[1],
                    "ci95_high": ss[2],
                    "n_queries": ss[3],
                },
                {
                    "analysis_tier": "full_policy",
                    "metric": metric,
                    "operator": operator,
                    "horizon": horizon,
                    "estimand": "representation_minus_search_H3abs",
                    "mean": ds[0],
                    "ci95_low": ds[1],
                    "ci95_high": ds[2],
                    "n_queries": ds[3],
                },
            ])

for metric in [
    "ndcg10",
    "mrr10",
    "recall10",
]:
    for operator in OPERATORS:
        for horizon in LONG_HORIZONS:
            rep, sea = query_mechanism_values(
                metric,
                operator,
                horizon,
                long_only=True,
            )

            rs = boot(rep)
            ss = boot(sea)
            ds = boot(rep - sea)

            rows.extend([
                {
                    "analysis_tier": "h50_subset",
                    "metric": metric,
                    "operator": operator,
                    "horizon": horizon,
                    "estimand": "representation_H3abs",
                    "mean": rs[0],
                    "ci95_low": rs[1],
                    "ci95_high": rs[2],
                    "n_queries": rs[3],
                },
                {
                    "analysis_tier": "h50_subset",
                    "metric": metric,
                    "operator": operator,
                    "horizon": horizon,
                    "estimand": "search_H3abs",
                    "mean": ss[0],
                    "ci95_low": ss[1],
                    "ci95_high": ss[2],
                    "n_queries": ss[3],
                },
                {
                    "analysis_tier": "h50_subset",
                    "metric": metric,
                    "operator": operator,
                    "horizon": horizon,
                    "estimand": "representation_minus_search_H3abs",
                    "mean": ds[0],
                    "ci95_low": ds[1],
                    "ci95_high": ds[2],
                    "n_queries": ds[3],
                },
            ])

summary = pd.DataFrame(rows)

SUMMARY_PATH = (
    OUT / "v028_h50_query_bootstrap.csv"
)

summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

display(
    summary.round(6)
)

primary = summary[
    (summary.analysis_tier == "h50_subset")
    & (summary.metric == "ndcg10")
    & (summary.operator == "recursive")
    & (summary.horizon == 50)
    & (
        summary.estimand
        == "representation_minus_search_H3abs"
    )
].iloc[0]

PRIMARY_SUPPORTED = bool(
    primary.ci95_low > 0
)

print(
    "PRIMARY H=50 AGENTIC-BRIDGE SUPPORTED:",
    PRIMARY_SUPPORTED,
)
print(
    primary.to_dict()
)

In [ ]:
# Cell 20 — Family robustness across all long horizons.
family_rows = []

for operator in OPERATORS:
    for horizon in LONG_HORIZONS:
        x = endpoints[
            (endpoints.operator == operator)
            & (endpoints.horizon == horizon)
            & (
                endpoints.trajectory_tier
                == "long_horizon"
            )
        ].copy()

        col = "ndcg10_H3_abs_slope"

        fam = x.groupby(
            [
                "query_id",
                "mechanism",
                "method",
            ],
            as_index=False,
        )[col].mean()

        for family in [
            "mean",
            "softmax",
        ]:
            r = (
                fam[
                    (
                        fam.mechanism
                        == "representation"
                    )
                    & (
                        fam.method
                        == family
                    )
                ]
                .set_index("query_id")
                .loc[VAL_IDS, col]
                .to_numpy(float)
            )

            s = (
                fam[
                    (
                        fam.mechanism
                        == "search_effort"
                    )
                    & (
                        fam.method
                        == family
                    )
                ]
                .set_index("query_id")
                .loc[VAL_IDS, col]
                .to_numpy(float)
            )

            st = boot(r - s)

            family_rows.append({
                "operator": operator,
                "horizon": horizon,
                "family": family,
                "mean": st[0],
                "ci95_low": st[1],
                "ci95_high": st[2],
                "n_queries": st[3],
            })

        piv = fam.pivot_table(
            index=[
                "query_id",
                "mechanism",
            ],
            columns="method",
            values=col,
        )

        assert {
            "mean",
            "softmax",
        }.issubset(piv.columns)

        piv["balanced"] = (
            0.5 * piv["mean"]
            + 0.5 * piv["softmax"]
        )

        rb = (
            piv.xs(
                "representation",
                level="mechanism",
            )
            .loc[VAL_IDS, "balanced"]
            .to_numpy(float)
        )

        sb = (
            piv.xs(
                "search_effort",
                level="mechanism",
            )
            .loc[VAL_IDS, "balanced"]
            .to_numpy(float)
        )

        st = boot(rb - sb)

        family_rows.append({
            "operator": operator,
            "horizon": horizon,
            "family": "equal_family",
            "mean": st[0],
            "ci95_low": st[1],
            "ci95_high": st[2],
            "n_queries": st[3],
        })

family_summary = pd.DataFrame(
    family_rows
)

FAMILY_PATH = (
    OUT / "v028_h50_family_robustness.csv"
)

family_summary.to_csv(
    FAMILY_PATH,
    index=False,
)

display(
    family_summary.round(6)
)

In [ ]:
# Cell 21 — H=50 persistence / interaction / late-gap / peak-round diagnostics.
def paired_query_delta(
    operator,
    horizon,
    metric="ndcg10",
):
    rep, sea = query_mechanism_values(
        metric,
        operator,
        horizon,
        long_only=True,
    )
    return rep - sea

interaction_rows = []

for metric in [
    "ndcg10",
    "mrr10",
    "recall10",
]:
    d = {
        (op, h): paired_query_delta(
            op,
            h,
            metric,
        )
        for op in OPERATORS
        for h in LONG_HORIZONS
    }

    contrasts = {
        "recursive_minus_anchored_at_h8":
            d[("recursive", 8)]
            - d[("anchored", 8)],

        "recursive_minus_anchored_at_h24":
            d[("recursive", 24)]
            - d[("anchored", 24)],

        "recursive_minus_anchored_at_h50":
            d[("recursive", 50)]
            - d[("anchored", 50)],

        "h24_minus_h8_recursive":
            d[("recursive", 24)]
            - d[("recursive", 8)],

        "h40_minus_h8_recursive":
            d[("recursive", 40)]
            - d[("recursive", 8)],

        "h50_minus_h8_recursive":
            d[("recursive", 50)]
            - d[("recursive", 8)],

        "h50_minus_h24_recursive":
            d[("recursive", 50)]
            - d[("recursive", 24)],

        "h50_minus_h40_recursive":
            d[("recursive", 50)]
            - d[("recursive", 40)],

        "h50_minus_h8_anchored":
            d[("anchored", 50)]
            - d[("anchored", 8)],

        "operator_x_h50_difference_in_differences":
            (
                d[("recursive", 50)]
                - d[("recursive", 8)]
                - (
                    d[("anchored", 50)]
                    - d[("anchored", 8)]
                )
            ),
    }

    for name, arr in contrasts.items():
        st = boot(arr)

        interaction_rows.append({
            "metric": metric,
            "contrast": name,
            "mean": st[0],
            "ci95_low": st[1],
            "ci95_high": st[2],
            "n_queries": st[3],
        })

interactions = pd.DataFrame(
    interaction_rows
)

INTERACTION_PATH = (
    OUT / "v028_h50_interactions.csv"
)

interactions.to_csv(
    INTERACTION_PATH,
    index=False,
)

display(
    interactions.round(6)
)

late_frames = []

for p in tqdm(
    parts,
    desc="Reading H8/H50 late-gap slices",
):
    x = pd.read_parquet(
        p,
        columns=[
            "query_id",
            "operator",
            "mechanism",
            "iteration",
            "trajectory_tier",
            "ndcg10_abs_gap",
        ],
    )

    x = x[
        (
            x.trajectory_tier
            == "long_horizon"
        )
        & (
            x.iteration.isin(
                [8, 50]
            )
        )
    ]

    g = (
        x.groupby(
            [
                "query_id",
                "operator",
                "mechanism",
                "iteration",
            ],
            as_index=False,
        )["ndcg10_abs_gap"]
        .mean()
    )

    late_frames.append(g)

late_q = pd.concat(
    late_frames,
    ignore_index=True,
)

late_p = late_q.pivot_table(
    index=[
        "query_id",
        "operator",
        "mechanism",
    ],
    columns="iteration",
    values="ndcg10_abs_gap",
)

late_p["round50_minus_round8"] = (
    late_p[50]
    - late_p[8]
)

late_rows = []

for op in OPERATORS:
    rep = (
        late_p.xs(
            (
                op,
                "representation",
            ),
            level=(
                "operator",
                "mechanism",
            ),
        )
        .loc[
            VAL_IDS,
            "round50_minus_round8",
        ]
        .to_numpy(float)
    )

    sea = (
        late_p.xs(
            (
                op,
                "search_effort",
            ),
            level=(
                "operator",
                "mechanism",
            ),
        )
        .loc[
            VAL_IDS,
            "round50_minus_round8",
        ]
        .to_numpy(float)
    )

    for name, arr in [
        (
            "representation_round50_minus_round8_abs_gap",
            rep,
        ),
        (
            "search_round50_minus_round8_abs_gap",
            sea,
        ),
        (
            "paired_rep_minus_search_late_gap_growth",
            rep - sea,
        ),
    ]:
        st = boot(arr)

        late_rows.append({
            "operator": op,
            "estimand": name,
            "mean": st[0],
            "ci95_low": st[1],
            "ci95_high": st[2],
            "n_queries": st[3],
        })

late_summary = pd.DataFrame(
    late_rows
)

LATE_PATH = (
    OUT / "v028_h50_late_gap_summary.csv"
)

late_summary.to_csv(
    LATE_PATH,
    index=False,
)

display(
    late_summary.round(6)
)

peak = endpoints[
    (
        endpoints.trajectory_tier
        == "long_horizon"
    )
    & (
        endpoints.horizon
        == 50
    )
].copy()

peak_summary = (
    peak.groupby(
        [
            "operator",
            "mechanism",
        ]
    )["ndcg10_peak_round"]
    .agg([
        "mean",
        "median",
        "std",
        "count",
    ])
    .reset_index()
)

PEAK_PATH = (
    OUT / "v028_h50_peak_gap_round_summary.csv"
)

peak_summary.to_csv(
    PEAK_PATH,
    index=False,
)

display(
    peak_summary.round(4)
)

In [ ]:
# Cell 22 — Regime composition across H4...H50.
regime_rows = []

for operator in OPERATORS:
    for horizon in LONG_HORIZONS:
        for mech in [
            "representation",
            "search_effort",
        ]:
            h = endpoints[
                (
                    endpoints.operator
                    == operator
                )
                & (
                    endpoints.horizon
                    == horizon
                )
                & (
                    endpoints.mechanism
                    == mech
                )
                & (
                    endpoints.trajectory_tier
                    == "long_horizon"
                )
            ][
                "ndcg10_H3_abs_slope"
            ].to_numpy(float)

            regime_rows.append({
                "operator": operator,
                "horizon": horizon,
                "mechanism": mech,
                "epsilon": EPS_PRIMARY,
                "stable_null_fraction": float(
                    (
                        np.abs(h)
                        <= EPS_PRIMARY
                    ).mean()
                ),
                "amplifying_fraction": float(
                    (
                        h
                        > EPS_PRIMARY
                    ).mean()
                ),
                "contracting_fraction": float(
                    (
                        h
                        < -EPS_PRIMARY
                    ).mean()
                ),
            })

regimes = pd.DataFrame(
    regime_rows
)

REGIME_PATH = (
    OUT / "v028_h50_regime_summary.csv"
)

regimes.to_csv(
    REGIME_PATH,
    index=False,
)

display(
    regimes.round(6)
)

In [ ]:
# Cell 23 — Frozen primary H=50 gate.
primary_family = family_summary[
    (
        family_summary.operator
        == "recursive"
    )
    & (
        family_summary.horizon
        == 50
    )
].copy()

long_ndcg_cells = summary[
    (
        summary.analysis_tier
        == "h50_subset"
    )
    & (
        summary.metric
        == "ndcg10"
    )
    & (
        summary.estimand
        == "representation_minus_search_H3abs"
    )
][
    [
        "operator",
        "horizon",
        "mean",
        "ci95_low",
        "ci95_high",
    ]
].to_dict(
    "records"
)

primary_metric_cells = summary[
    (
        summary.analysis_tier
        == "h50_subset"
    )
    & (
        summary.operator
        == "recursive"
    )
    & (
        summary.horizon
        == 50
    )
    & (
        summary.estimand
        == "representation_minus_search_H3abs"
    )
][
    [
        "metric",
        "mean",
        "ci95_low",
        "ci95_high",
    ]
].to_dict(
    "records"
)

GATE = {
    "status": (
        "ARC_V028_H50_A100_BATCHED_"
        "AGENTIC_BRIDGE_ANALYZED"
    ),
    "protocol_sha256": PROTOCOL_SHA,
    "dataset": DATASET_NAME,
    "encoder": ENCODER_NAME,
    "classification": (
        PROTOCOL["classification"]
    ),
    "execution_backend": EXECUTION_BACKEND,
    "n_validation_queries": len(VAL_IDS),
    "primary_new_estimand": (
        PROTOCOL["primary_new_estimand"]
    ),
    "primary_result": {
        "mean": float(
            primary["mean"]
        ),
        "ci95": [
            float(
                primary["ci95_low"]
            ),
            float(
                primary["ci95_high"]
            ),
        ],
        "ci_excludes_zero_positive": (
            PRIMARY_SUPPORTED
        ),
    },
    "primary_recursive_h50_agentic_bridge_supported": (
        PRIMARY_SUPPORTED
    ),
    "long_horizon_ndcg_operator_cells": (
        long_ndcg_cells
    ),
    "recursive_h50_metric_cells": (
        primary_metric_cells
    ),
    "recursive_h50_family_robustness": (
        primary_family.to_dict(
            "records"
        )
    ),
    "negative_null_saturation_reversal_retained": True,
    "validation_retuning_performed": False,
    "not_a_full_llm_agent_experiment": True,
    "completed_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
}

GATE_PATH = (
    OUT / "v028_primary_h50_agentic_bridge_gate.json"
)

GATE_PATH.write_text(
    json.dumps(
        GATE,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        GATE,
        indent=2,
    )
)

In [ ]:
# Cell 24 — Final report and artifact hashes.
REPORT = {
    **GATE,
    "endpoint_path": str(
        ENDPOINT_PATH
    ),
    "checkpoint_manifest_path": str(
        CHECKPOINT_MANIFEST_PATH
    ),
    "summary_path": str(
        SUMMARY_PATH
    ),
    "family_summary_path": str(
        FAMILY_PATH
    ),
    "interaction_path": str(
        INTERACTION_PATH
    ),
    "late_horizon_summary_path": str(
        LATE_PATH
    ),
    "peak_round_summary_path": str(
        PEAK_PATH
    ),
    "regime_path": str(
        REGIME_PATH
    ),
    "backend_audit_path": str(
        AUDIT_PATH
    ),
    "scientific_claim_scope": (
        "Post-v0.27 NQ-GTE operator/horizon stress test. "
        "The long-horizon tier evaluates anchored and recursive "
        "retrieval-feedback trajectories through 50 updates as a "
        "controlled bridge toward Agentic IR / Agentic RAG. "
        "It does not instantiate an LLM planner, adaptive tool policy, "
        "dynamic stopping policy, memory manager, or full agent environment."
    ),
}

REPORT_PATH = (
    OUT / "v028_final_h50_agentic_bridge_report.json"
)

REPORT_PATH.write_text(
    json.dumps(
        REPORT,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

artifact_paths = [
    PROTOCOL_PATH,
    SPLIT_PATH,
    POLICY_PATH,
    LONG_POLICY_PATH,
    AUDIT_PATH,
    CHECKPOINT_MANIFEST_PATH,
    ENDPOINT_PATH,
    SUMMARY_PATH,
    FAMILY_PATH,
    INTERACTION_PATH,
    LATE_PATH,
    PEAK_PATH,
    REGIME_PATH,
    GATE_PATH,
    REPORT_PATH,
]

hash_df = pd.DataFrame([
    {
        "file": p.name,
        "bytes": p.stat().st_size,
        "sha256": sha256_file(p),
    }
    for p in artifact_paths
])

HASH_PATH = (
    OUT / "V028_H50_ARTIFACT_SHA256.csv"
)

hash_df.to_csv(
    HASH_PATH,
    index=False,
)

display(
    hash_df
)

print("=" * 96)
print(
    "ARC-v0.28 H=50 A100-BATCHED AGENTIC-IR BRIDGE — COMPLETE"
)
print("OUT:", OUT)
print("EXECUTION BACKEND:", EXECUTION_BACKEND)
print(
    "PRIMARY RECURSIVE H50 SUPPORTED:",
    PRIMARY_SUPPORTED,
)
print("=" * 96)

## What to send back

After completion, send the executed notebook or these outputs:

1. `v028_h50_query_bootstrap.csv`
2. `v028_h50_family_robustness.csv`
3. `v028_h50_interactions.csv`
4. `v028_h50_late_gap_summary.csv`
5. `v028_h50_peak_gap_round_summary.csv`
6. `v028_primary_h50_agentic_bridge_gate.json`
7. `v028_final_h50_agentic_bridge_report.json`
8. `v028_fit_execution_backend_audit.json`

The primary result is:

> **long-horizon subset, recursive operator, H=50, nDCG@10 representation-minus-search H3abs**

and its paired query-bootstrap 95% CI.

The key scientific secondary question is whether the ordering **persists, saturates, or reverses from H=8 to H=50**.

Do not change the frozen severity match, long-horizon subset, operators, horizons, endpoints, backend-equivalence gate, or success criterion after validation trajectories are observed.